In [13]:

# Supervised Learning on ECB Communications: Predicting Rate Decisions and Market Reactions
# ----------------------------------------------------------------------------
# This script mirrors a common FOMC pipeline, adapted to the ECB:
# - Texts: ECB statements / press statements (and optionally press conference statements), one row per event-date
# - Labels: +1 (rate hike), -1 (rate cut) based on MRO rate changes; optional "no change" class
# - Models: TF-IDF + Logistic (elastic-net), and TF-IDF + ElasticNet (regression)
# - Optional: compare with SentenceTransformer embeddings
# - Optional: supervised learning of same-day EuroStoxx returns on statement text
#
# INPUT FILES (adjust paths as needed):
#   data/ecb_statements.csv   -> columns: date,text
#   data/ecb_policy_rates.csv -> columns: date,mro_rate  (Main Refinancing Operations policy rate, in percent)
#   data/eurostoxx_daily.csv  -> columns: date,ret      (EuroStoxx 50 daily simple returns), optional
#
# NOTE: If you already use the skfin helpers (coefs_plot, show_text), they are imported if available.
#       Otherwise, simple fallbacks are provided.

from datetime import date
from pathlib import Path

import pandas as pd
import numpy as np
from pandas.tseries.offsets import BDay

import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.pipeline import Pipeline

# -----------------------------------------------------------------------------
# Optional helpers from skfin; fall back gracefully if not available
try:
    from skfin.text import coefs_plot, show_text
    from skfin.plot import line
    _HAS_SKFIN = True
except Exception:
    _HAS_SKFIN = False
    def coefs_plot(df_coef, title="Coefficients"):
        # Minimal fallback: barh of top/bottom 10
        s = df_coef.squeeze()
        top = s.nlargest(10)
        bot = s.nsmallest(10)
        fig, ax = plt.subplots(figsize=(8, 6))
        both = pd.concat([bot, top])
        both.sort_values().plot(kind="barh", ax=ax)
        ax.set_title(title)
        plt.tight_layout()

    def show_text(df_text, lexica=None, n=None):
        # Minimal fallback: print text and (optionally) highlight lexicon words
        for idx, row in df_text.iterrows():
            print(f"=== {idx} ===")
            print(row["text"][:800] + ("..." if len(row["text"]) > 800 else ""))
            if lexica:
                print("\nTop positive:\n", list(lexica.get("positive", []).index))
                print("Top negative:\n", list(lexica.get("negative", []).index))
            print()

    def line(df, sort=False, ax=None, title=None):
        ax = ax or plt.gca()
        df.plot(ax=ax)
        if title:
            ax.set_title(title)

# -----------------------------------------------------------------------------
# Paths (you can edit)
DATA_DIR = Path(r"C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset")
ECB_SPEECHES_CSV = DATA_DIR / "ecb_speeches_clean_minimal.csv"
ECB_POLICY_RATES_CSV = DATA_DIR / "ecb_policy_rates_daily_fake.csv"
EUROSTOXX_CSV = DATA_DIR / "eurostoxx_daily_fake.csv"  # optional


In [14]:
from pandas.tseries.offsets import BDay

def _normalize_index_to_date_index(df, date_col="date"):
    """Met l'index au format DatetimeIndex normalisé (00:00), sans TZ, unique et trié."""
    if date_col in df.columns:
        s = pd.to_datetime(df[date_col], errors="coerce")
    else:
        # Index déjà datetime ?
        s = pd.to_datetime(df.index, errors="coerce")
    s = s.dt.tz_localize(None)
    s = s.dt.normalize()
    df = df.copy()
    df.index = s
    df = df[~df.index.isna()]
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="first")]
    return df

def load_ecb_statements(path=ECB_SPEECHES_CSV):
    df = pd.read_csv(path)
    assert "date" in df.columns and "text" in df.columns, "Le CSV statements doit avoir 'date' et 'text'"
    df = _normalize_index_to_date_index(df, "date")
    return df[["text"]]

def load_ecb_mro(path=ECB_POLICY_RATES_CSV):
    df = pd.read_csv(path)
    assert "date" in df.columns and "mro_rate" in df.columns, "Le CSV MRO doit avoir 'date' et 'mro_rate'"
    df["mro_rate"] = pd.to_numeric(df["mro_rate"], errors="coerce")
    df = df.dropna(subset=["mro_rate"])
    df = _normalize_index_to_date_index(df, "date")
    return df[["mro_rate"]]

def compute_rate_change_labels(mro: pd.DataFrame):
    """
    Fonctionne avec:
      - série *quotidienne* (forward-filled) OU
      - *seulement* les dates de changement.
    Retourne un DataFrame 'events' indexé par dates EFFECTIVES de changement,
    avec colonne 'change' ∈ {-1,+1}.
    """
    assert isinstance(mro.index, pd.DatetimeIndex)
    mro = mro.sort_index().copy()

    # Si les dates intermédiaires manquent, cette diff marche (delta NaN hors changements).
    # Si on a une série quotidienne forward-filled, ça marche aussi.
    mro["delta"] = mro["mro_rate"].diff()

    # Garde uniquement les vrais changements
    ev = mro.loc[mro["delta"].fillna(0) != 0, ["delta"]].copy()
    ev["change"] = np.sign(ev["delta"]).astype(int)
    ev = ev.drop(columns=["delta"])

    # Diagnostics
    up = ev.index[ev["change"] == 1]
    dw = ev.index[ev["change"] == -1]
    print(f"[LABELS] changements effectifs: total={len(ev)} | hikes={len(up)} | cuts={len(dw)} "
          f"| span={mro.index.min().date()}→{mro.index.max().date()}")
    return ev  # DataFrame: index=dates de changement, col 'change' ∈ {-1,+1}

def align_statements_and_labels(statements: pd.DataFrame,
                                events: pd.DataFrame,
                                back_bdays=1,
                                fwd_bdays=2):
    """
    Aligne chaque statement (date S) vers le changement effectif le plus proche dans la fenêtre:
      [S - back_bdays*BD, S + fwd_bdays*BD]
    Priorité au futur (S→S+1BD→S+2BD), sinon regarde S-1BD.
    Conflits (deux events dans la fenêtre): on prend le plus proche; en cas d'égalité, priorité au futur.
    Retourne un DF indexé par S (dates de statements) avec colonnes: text, change.
    """
    assert isinstance(statements.index, pd.DatetimeIndex)
    assert isinstance(events.index, pd.DatetimeIndex)
    statements = statements.sort_index()
    events = events.sort_index()

    # Prépare des sets pour lookup rapide
    ev_idx = pd.DatetimeIndex(events.index)

    # Décalages (ordre = priorité)
    forward_lags = [0] + [i for i in range(1, fwd_bdays+1)]
    backward_lags = [i for i in range(1, back_bdays+1)]

    matches_idx = []
    matches_change = []

    for s in statements.index:
        candidates = []
        # futur / même jour d'abord
        for k in forward_lags:
            target = (s + BDay(k)) if k > 0 else s
            if target in ev_idx:
                candidates.append((abs(k), +1, target))  # +1 = priorité futur/présent
        # puis passé (si rien trouvé)
        if not candidates:
            for k in backward_lags:
                target = s - BDay(k)
                if target in ev_idx:
                    candidates.append((abs(k), -1, target))  # -1 = passé
        if candidates:
            # tri: distance min puis priorité futur (clé (dist, -priority_flag))
            # ici priority_flag=+1 (futur) doit gagner à égalité, donc on trie par (-priority_flag)
            best = sorted(candidates, key=lambda x: (x[0], -x[1]))[0]
            chosen_date = best[2]
            matches_idx.append(s)
            matches_change.append(int(events.loc[chosen_date, "change"]))

    if not matches_idx:
        print("[ALIGN] aucun statement mappé — élargir la fenêtre?")
        return pd.DataFrame(columns=["text","change"], index=pd.DatetimeIndex([], name=statements.index.name))

    df_lbl = statements.loc[matches_idx].copy()
    df_lbl["change"] = matches_change
    df_lbl = df_lbl.sort_index()

    # Diagnostics
    coverage = len(df_lbl) / len(statements) * 100 if len(statements) else 0.0
    vc = df_lbl["change"].value_counts().to_dict()
    earliest = df_lbl.index.min().date() if len(df_lbl) else "—"
    latest = df_lbl.index.max().date() if len(df_lbl) else "—"
    print(f"[ALIGN] labeled={len(df_lbl)} / {len(statements)} ({coverage:.1f}%) | "
          f"class_balance={vc} | span={earliest}→{latest} | "
          f"window=[-{back_bdays}BD, +{fwd_bdays}BD]")

    return df_lbl


In [15]:
# -----------------------------------------------------------------------------
# Load data
def load_ecb_statements(path=ECB_SPEECHES_CSV) :
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df.sort_values("date").drop_duplicates("date").set_index("date")
    # Ensure text column exists
    assert "text" in df.columns, "ecb_statements.csv must have a 'text' column"
    return df[["text"]]

def load_ecb_mro(path=ECB_POLICY_RATES_CSV):
    # Expect one row per effective date where a *new* MRO rate becomes valid.
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df.sort_values("date").drop_duplicates("date").set_index("date")
    assert "mro_rate" in df.columns, "ecb_policy_rates.csv must have 'mro_rate' column (percent)"
    return df[["mro_rate"]]

def compute_rate_change_labels(mro: pd.DataFrame):
    # Compute the discrete change in MRO at each effective date
    mro_sorted = mro.sort_index()
    mro_sorted["mro_rate_prev"] = mro_sorted["mro_rate"].shift(1)
    mro_sorted["delta"] = mro_sorted["mro_rate"] - mro_sorted["mro_rate_prev"]
    # Label +1 for hikes, -1 for cuts at the date the new rate becomes effective
    labels = pd.Series(np.sign(mro_sorted["delta"]).fillna(0).astype(int), index=mro_sorted.index, name="change")
    # Only keep actual change events (+/-1); zero changes may correspond to the first obs or non-change updates
    up_dates = labels[labels == 1].index
    dw_dates = labels[labels == -1].index
    return up_dates, dw_dates, labels

def align_statements_and_labels(statements: pd.DataFrame, up_dates, dw_dates):
    # Some ECB statement dates may not exactly match the rate effective date.
    # Commonly, statement date == Governing Council meeting date; MRO effective date may be next day.
    # For supervision, we use statement dates and map them to +/-1 if within a small window of an effective change.
    s_idx = statements.index
    up = s_idx.intersection(up_dates)              # direct match
    dw = s_idx.intersection(dw_dates)

    # If few matches, also try aligning when the effective change is next business day
    up_plus1 = s_idx.intersection((pd.DatetimeIndex(up_dates) - BDay(1)))
    dw_plus1 = s_idx.intersection((pd.DatetimeIndex(dw_dates) - BDay(1)))

    up_all = up.union(up_plus1).unique()
    dw_all = dw.union(dw_plus1).unique()

    # Build binary labels (+1/-1); "no change" days are simply unused in binary fit
    df_lbl = pd.concat([
        statements.loc[up_all].assign(change=1),
        statements.loc[dw_all].assign(change=-1),
    ]).sort_index()

    return df_lbl

In [16]:

# -----------------------------------------------------------------------------
# Main: supervised learning to classify rate changes from ECB statements
def train_ecb_tfidf_logreg(statements: pd.DataFrame, up_dates, dw_dates):
    df_lbl = align_statements_and_labels(statements, up_dates, dw_dates)
    X, y = df_lbl["text"], df_lbl["change"]

    est = Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(
                vocabulary=None,
                ngram_range=(1, 3),
                max_features=500,
                stop_words="english",
                token_pattern=r"\\b[a-zA-Z]{3,}\\b",
            )),
            ("log1p", FunctionTransformer(np.log1p)),
            ("reg", LogisticRegression(
                C=1, l1_ratio=0.35, penalty="elasticnet", solver="saga", max_iter=1000
            )),
        ]
    )
    est.fit(X, y)

    vocab_ = pd.Series(est.named_steps["tfidf"].vocabulary_).sort_values().index
    interpret_coef = pd.DataFrame(np.transpose(est.named_steps["reg"].coef_), index=vocab_)

    return est, interpret_coef

def train_ecb_tfidf_elasticnet(statements: pd.DataFrame, up_dates, dw_dates):
    df_lbl = align_statements_and_labels(statements, up_dates, dw_dates)
    X, y = df_lbl["text"], df_lbl["change"]

    est = Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(
                vocabulary=None,
                ngram_range=(1, 3),
                max_features=500,
                stop_words="english",
                token_pattern=r"\\b[a-zA-Z]{3,}\\b",
            )),
            ("log1p", FunctionTransformer(np.log1p)),
            ("reg", ElasticNet(alpha=0.01)),
        ]
    )
    est.fit(X, y)

    vocab_ = pd.Series(est.named_steps["tfidf"].vocabulary_).sort_values().index
    interpret_coef = pd.DataFrame(np.transpose(est.named_steps["reg"].coef_), index=vocab_)

    return est, interpret_coef

def implied_rate_series(est, statements: pd.DataFrame, up_dates, dw_dates, title="ECB implied rate (with forward information)"):
    pred_tfidf = (
        pd.Series(est.predict(statements["text"]), index=statements.index)
        .resample("B")
        .last()
        .ffill()
    )
    fig, ax = plt.subplots(figsize=(9, 6))
    df_plot = (
        pred_tfidf.rename("implied rate")
        .to_frame()
        .join(pd.Series(1, index=pd.to_datetime(up_dates)).reindex(pred_tfidf.index).fillna(0).rename("up"))
        .join(pd.Series(-1, index=pd.to_datetime(dw_dates)).reindex(pred_tfidf.index).fillna(0).rename("down"))
    )
    line(df_plot, sort=False, ax=ax, title=title)
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    plt.tight_layout()
    return pred_tfidf


In [17]:

# -----------------------------------------------------------------------------
# Optional: sentence-transformer comparison
def compare_tfidf_sbert(statements: pd.DataFrame, pred_tfidf: pd.Series, model_name="all-distilroberta-v1"):
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as e:
        print("SentenceTransformers not available. Skipping SBERT comparison.")
        return None

    m = SentenceTransformer(model_name, device="cpu", trust_remote_code=True)
    X_sbert = m.encode(statements["text"].values, batch_size=2)
    df = pd.DataFrame(X_sbert, index=statements.index)

    # Simple regression to map embeddings to +/-1 labels using same event set as pred_tfidf index
    # For illustration only (full-sample fit)
    en = ElasticNet(alpha=0.01).fit(df, np.sign(pred_tfidf.reindex(df.index).fillna(0)).values)
    pred_sbert = pd.Series(en.predict(df), index=df.index).resample("B").last().ffill()

    corr_ = pd.concat({"sbert": pred_sbert, "tfidf": pred_tfidf}, axis=1).corr().iloc[0, 1]
    print(f"Correlation between SBERT and TF-IDF implied series: {corr_:.2f}")

    fig, ax = plt.subplots(figsize=(9, 4))
    pd.concat({"sbert": pred_sbert, "tfidf": pred_tfidf}, axis=1).pipe(lambda x: x.div(x.std())).plot(ax=ax)
    ax.set_title("Standardized implied series: SBERT vs TF-IDF")
    plt.tight_layout()
    return corr_

In [18]:
import csv
import io

def _sniff_csv_params(path, sample_bytes=65536):
    with open(path, "rb") as f:
        raw = f.read(sample_bytes)
    # Essaye d'inférer l'encodage simple: BOM UTF-8 ?
    encoding = "utf-8-sig" if raw.startswith(b"\xef\xbb\xbf") else "utf-8"
    text = raw.decode(encoding, errors="replace")
    try:
        dialect = csv.Sniffer().sniff(text, delimiters=[",",";","\t","|"])
        sep = dialect.delimiter
        quotechar = dialect.quotechar if dialect.quotechar else '"'
    except Exception:
        sep, quotechar = None, '"'
    return encoding, sep, quotechar

def load_ecb_statements(path=ECB_SPEECHES_CSV):
    print(f"\n[LOAD] ECB statements from: {path}")
    # 1) Sniff
    enc, sep, quotechar = _sniff_csv_params(path)
    print(f"[SNIFF] encoding={enc} | sep={'auto' if sep is None else repr(sep)} | quotechar={repr(quotechar)}")

    # 2) Essais successifs
    tries = [
        dict(encoding=enc, sep=sep, engine="python", quotechar=quotechar, escapechar="\\"),
        dict(encoding=enc, sep=",", engine="python", quotechar='"', escapechar="\\"),
        dict(encoding=enc, sep=";", engine="python", quotechar='"', escapechar="\\"),
        dict(encoding="latin-1", sep=sep, engine="python", quotechar=quotechar, escapechar="\\"),
    ]
    last_err = None
    for i, kw in enumerate(tries, 1):
        try:
            df = pd.read_csv(path, **{k:v for k,v in kw.items() if v is not None})
            print(f"[READ OK] try#{i} -> shape={df.shape}")
            break
        except Exception as e:
            print(f"[READ FAIL] try#{i}: {e}")
            last_err = e
            df = None
    if df is None:
        raise last_err

    # Vérifs colonnes
    if "date" not in df.columns or "text" not in df.columns:
        # Parfois le séparateur est bon mais le header est cassé → on tente sans header et on renomme
        print("[INFO] 'date'/'text' non trouvés, tentative sans header...")
        df2 = pd.read_csv(path, header=None, **{k:v for k,v in tries[0].items() if v is not None})
        print(f"[READ NO-HEADER] shape={df2.shape}")
        # Heuristique: 1ère colonne date, dernière colonne texte (si beaucoup de colonnes)
        if df2.shape[1] >= 2:
            df = df2.rename(columns={0:"date", df2.shape[1]-1:"text"})
        else:
            raise AssertionError("Impossible d'identifier les colonnes 'date' et 'text'.")

    # Parse dates
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    bad = df["date"].isna().sum()
    if bad:
        print(f"[WARN] {bad} dates non parseables seront supprimées")
        df = df.dropna(subset=["date"])

    # Nettoyage texte minimal
    df["text"] = df["text"].astype(str)

    # Tri/uniques/index
    before = len(df)
    df = df.sort_values("date").drop_duplicates("date", keep="first").set_index("date")
    after = len(df)
    print(f"[OK] statements: shape={df.shape} | dup_dates_removed={before-after} | "
          f"range={df.index.min().date()} → {df.index.max().date()}")

    # Sanity rapide
    blank_txt = (df["text"].str.strip() == "").sum()
    if blank_txt:
        print(f"[WARN] {blank_txt} lignes avec texte vide")
    return df[["text"]]


In [25]:
# Smoke test end-to-end
stm = load_ecb_statements()
mro = load_ecb_mro()
up, dw, _ = compute_rate_change_labels(mro)
df_lbl = align_statements_and_labels(stm, up, dw)

print("\n[SUMMARY]")
print(f"  statements total = {len(stm)}")
print(f"  labeled (±1)     = {len(df_lbl)}")
print(f"  unlabeled        = {len(stm) - len(df_lbl)}")
print(f"  time span stm    = {stm.index.min().date()} → {stm.index.max().date()}")
if len(df_lbl):
    print(f"  time span lbl    = {df_lbl.index.min().date()} → {df_lbl.index.max().date()}")



[LOAD] ECB statements from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_speeches_clean_minimal.csv
[SNIFF] encoding=utf-8 | sep=',' | quotechar='"'
[READ OK] try#1 -> shape=(2939, 5)
[OK] statements: shape=(2249, 4) | dup_dates_removed=690 | range=1997-02-07 → 2025-09-30
[LABELS] changes: total=11 | hikes=8 | cuts=3 | span=2015-01-01→2025-10-31


ValueError: not enough values to unpack (expected 3, got 1)

In [24]:
# === DIAGNOSTICS RAPIDES ===
stm = load_ecb_statements()
mro = load_ecb_mro()

print("\n[MRO SUMMARY]")
print("shape:", mro.shape, "| span:", mro.index.min().date(), "→", mro.index.max().date())
print("nunique rates:", mro["mro_rate"].nunique())
print("head:\n", mro.head(5))
print("tail:\n", mro.tail(5))

# Détection d'événements
def _events_from_mro(mro):
    s = mro["mro_rate"].astype(float)
    delta = s.diff()
    ev = delta[delta.fillna(0) != 0].index
    return ev

ev_idx = _events_from_mro(mro)
print("\n[EVENTS] nb_effective_changes:", len(ev_idx))
print("first 5:", list(ev_idx[:5]))
print("last 5 :", list(ev_idx[-5:]))

# Comptage d'intersections sur différentes fenêtres
def _count_matches(stm_idx, ev_idx, back=2, fwd=3):
    from pandas.tseries.offsets import BDay
    stm_idx = pd.DatetimeIndex(stm_idx)
    ev_idx  = pd.DatetimeIndex(ev_idx)
    counts = {}
    for k in range(-back, fwd+1):
        if k == 0:
            target = ev_idx
        elif k > 0:
            target = ev_idx - BDay(k)  # statement jour J, effectif J+k  ⇒ match sur J = ev - kBD
        else:
            target = ev_idx - BDay(k)  # k négatif ⇒ ev - (-k) = ev + |k|
        counts[k] = len(stm_idx.intersection(target))
    return counts

match_counts = _count_matches(stm.index, ev_idx, back=2, fwd=3)
print("\n[MATCH COUNTS by offset k where statement_date = effective_date - kBD]")
for k in sorted(match_counts):
    print(f"k={k:+}: {match_counts[k]}")



[LOAD] ECB statements from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_speeches_clean_minimal.csv
[SNIFF] encoding=utf-8 | sep=',' | quotechar='"'
[READ OK] try#1 -> shape=(2939, 5)
[OK] statements: shape=(2249, 4) | dup_dates_removed=690 | range=1997-02-07 → 2025-09-30

[MRO SUMMARY]
shape: (2827, 1) | span: 2015-01-01 → 2025-10-31
nunique rates: 6
head:
             mro_rate
date                
2015-01-01      0.75
2015-01-02      0.75
2015-01-05      0.75
2015-01-06      0.75
2015-01-07      0.75
tail:
             mro_rate
date                
2025-10-27       2.0
2025-10-28       2.0
2025-10-29       2.0
2025-10-30       2.0
2025-10-31       2.0

[EVENTS] nb_effective_changes: 11
first 5: [Timestamp('2015-09-10 00:00:00'), Timestamp('2018-06-14 00:00:00'), Timestamp('2019-02-21 00:00:00'), Timestamp('2019-10-31 00:00:00'), Timestamp('2020-07-09 00:00:00')]
last 5 : [Timestamp('2022-08-04 00:00:00'), Timestamp('2023-04-13 00:00:00'), Timestamp('2023-12-21 0

In [27]:
from pandas.tseries.offsets import BDay

def compute_rate_change_events(mro: pd.DataFrame):
    """
    Détecte les dates EFFECTIVES de changement (delta != 0) dans une série MRO quotidienne
    (ou uniquement aux dates de changement). Retourne un DataFrame 'events' indexé par
    date effective avec 'change' ∈ {-1,+1}.
    """
    assert isinstance(mro.index, pd.DatetimeIndex), "`mro` doit avoir un DatetimeIndex"
    mro = mro.sort_index().copy()
    mro["mro_rate"] = pd.to_numeric(mro["mro_rate"], errors="coerce")
    mro = mro.dropna(subset=["mro_rate"])

    delta = mro["mro_rate"].diff()
    ev = delta[delta.fillna(0) != 0]
    events = pd.DataFrame(index=ev.index)
    events["change"] = np.sign(ev.values).astype(int)

    print(f"[EVENTS] changes: total={len(events)} | hikes={(events['change']==1).sum()} "
          f"| cuts={(events['change']==-1).sum()} | span={mro.index.min().date()}→{mro.index.max().date()}")
    return events


def map_events_to_statements(statements: pd.DataFrame,
                             events: pd.DataFrame,
                             back_bdays=2,   # chercher un statement jusqu'à 2 jours ouvrés AVANT l'event
                             fwd_bdays=1,    # et jusqu'à 1 jour ouvré APRÈS
                             prefer_past=True):
    """
    Pour chaque EVENT (date effective E), cherche le statement S le plus proche dans la fenêtre
    [E - back_bdays*BD, E + fwd_bdays*BD]. Par défaut on **préfère le passé** (S ≤ E),
    car l'effectif BCE est souvent J+1 par rapport au jour de réunion.

    Si deux events mappent sur le même statement, on garde celui avec la distance |BD| la plus faible.

    Retourne un DF 'df_lbl' indexé par dates de STATEMENTS, avec colonnes: text, change.
    """
    assert isinstance(statements.index, pd.DatetimeIndex), "`statements` doit avoir un DatetimeIndex"
    assert isinstance(events.index, pd.DatetimeIndex), "`events` doit avoir un DatetimeIndex"

    s_idx = pd.DatetimeIndex(statements.index).sort_values()
    ev_idx = pd.DatetimeIndex(events.index).sort_values()

    # Prépare une table pour résolution des collisions: statement_date -> (dist_bd, priority, change)
    # priority: on met 1 si past (S<=E) quand prefer_past=True, 0 sinon; sert à casser les égalités.
    chosen = {}

    for e in ev_idx:
        # Construire la fenêtre business days autour de E
        window = set([e])
        for k in range(1, back_bdays+1):
            window.add(e - BDay(k))
        for k in range(1, fwd_bdays+1):
            window.add(e + BDay(k))

        # Intersections possibles avec des statements existants
        candidates = s_idx.intersection(pd.DatetimeIndex(sorted(window)))
        if len(candidates) == 0:
            continue

        # Choisir le S* minimal en distance business (|k|), préférence passé si égalité
        best_S = None
        best_key = None  # (dist, tie_break)
        for s in candidates:
            # calcule la distance business approx en comptant les BDays entre s et e
            # (utilise un compteur simple: on décale pas à pas; suffisant ici)
            if s == e:
                dist = 0
            elif s < e:
                # s avant e
                # compter le nombre de BDays de s à e (approx: boucle courte car fenêtre petite)
                k = 0
                cur = s
                while cur < e:
                    cur += BDay(1)
                    k += 1
                dist = k
            else:
                # s après e
                k = 0
                cur = s
                while cur > e:
                    cur -= BDay(1)
                    k += 1
                dist = k

            # tie-break: si prefer_past=True, on veut prioriser s<=e à égalité
            tie = 0
            if prefer_past:
                tie = 1 if s <= e else 0

            key = (dist, -tie)  # plus petit dist, puis tie le plus grand (donc passé gagne)
            if (best_key is None) or (key < best_key):
                best_key = key
                best_S = s

        if best_S is None:
            continue

        # Enregistre/resolve collisions
        change = int(events.loc[e, "change"])
        dist_choice, tie_choice = best_key
        prev = chosen.get(best_S)
        if prev is None or (best_key < prev[:2]):  # remplace si meilleur
            chosen[best_S] = (best_key[0], best_key[1], change)

    if not chosen:
        print(f"[MAP] Aucun event mappé à un statement dans la fenêtre "
              f"[-{back_bdays}BD, +{fwd_bdays}BD]. Essaie d’élargir la fenêtre.")
        return statements.iloc[0:0].copy()

    # Construire la table labellisée
    S_dates = sorted(chosen.keys())
    changes = [chosen[s][2] for s in S_dates]
    df_lbl = statements.loc[S_dates].copy()
    df_lbl["change"] = changes
    vc = df_lbl["change"].value_counts().to_dict()
    coverage = len(df_lbl) / len(statements) * 100 if len(statements) else 0.0
    print(f"[MAP] labeled={len(df_lbl)} / {len(statements)} ({coverage:.1f}%) | class_balance={vc} | "
          f"window=[-{back_bdays}BD, +{fwd_bdays}BD] (préférence passé={prefer_past})")
    return df_lbl


In [28]:
stm = load_ecb_statements()
mro = load_ecb_mro()

events = compute_rate_change_events(mro)

# Essaie d’abord la fenêtre standard J-2BD → J+1BD
df_lbl = map_events_to_statements(stm, events, back_bdays=2, fwd_bdays=1, prefer_past=True)

print("\n[SUMMARY]")
print("  statements total :", len(stm))
print("  labeled (±1)     :", len(df_lbl))
print("  unlabeled        :", len(stm) - len(df_lbl))
if len(df_lbl):
    print("  class balance    :", df_lbl['change'].value_counts().to_dict())
    print("  span (labeled)   :", df_lbl.index.min().date(), "→", df_lbl.index.max().date())



[LOAD] ECB statements from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_speeches_clean_minimal.csv
[SNIFF] encoding=utf-8 | sep=',' | quotechar='"'
[READ OK] try#1 -> shape=(2939, 5)
[OK] statements: shape=(2249, 4) | dup_dates_removed=690 | range=1997-02-07 → 2025-09-30
[EVENTS] changes: total=11 | hikes=8 | cuts=3 | span=2015-01-01→2025-10-31
[MAP] Aucun event mappé à un statement dans la fenêtre [-2BD, +1BD]. Essaie d’élargir la fenêtre.

[SUMMARY]
  statements total : 2249
  labeled (±1)     : 0
  unlabeled        : 2249


In [29]:
def inspect_statements_dataframe(df_statements_raw):
    print("\n[STATEMENTS] Colonnes disponibles:", list(df_statements_raw.columns))
    # Compte dates et période
    print("[STATEMENTS] N lignes:", len(df_statements_raw))
    if "date" in df_statements_raw.columns:
        d = pd.to_datetime(df_statements_raw["date"], errors="coerce")
        print("[STATEMENTS] Période:", d.min(), "→", d.max())

    # Aperçu de 3 lignes
    print("\n[HEAD]\n", df_statements_raw.head(3))

    # Si 'title' existe, inspecte des keywords typiques de politique monétaire
    if "title" in df_statements_raw.columns:
        t = df_statements_raw["title"].astype(str).str.lower()
        keys = {
            "monetary policy decision": t.str.contains("monetary policy decision"),
            "introductory statement": t.str.contains("introductory statement"),
            "press conference": t.str.contains("press conference"),
            "press release": t.str.contains("press release"),
            "governing council": t.str.contains("governing council"),
            "interest rate": t.str.contains("interest rate|policy rate|refinancing rate"),
        }
        print("\n[STATEMENTS] Comptes par mots-clés (dans 'title'):")
        for k, m in keys.items():
            print(f"  - {k:25s}: {int(m.sum())}")

    # Distribution jour de semaine (0=lundi … 6=dimanche) si on a date
    if "date" in df_statements_raw.columns:
        wd = pd.to_datetime(df_statements_raw["date"], errors="coerce").dt.dayofweek
        print("\n[STATEMENTS] Répartition par weekday (0=Lun ... 6=Dim):")
        print(wd.value_counts().sort_index())


In [23]:
# === ÉVÉNEMENTS ROBUSTES ===
def compute_rate_change_labels(mro: pd.DataFrame):
    """
    Retourne un DataFrame 'events' indexé par les dates EFFECTIVES de changement (jour où le nouveau taux devient effectif),
    avec 'change' ∈ {-1, +1}.
    Compatible avec série quotidienne forward-fill OU seulement les dates d'annonce/effective.
    """
    assert isinstance(mro.index, pd.DatetimeIndex)
    mro = mro.sort_index().copy()

    # Type numérique strict
    mro["mro_rate"] = pd.to_numeric(mro["mro_rate"], errors="coerce")
    mro = mro.dropna(subset=["mro_rate"])

    # Détecte les sauts
    delta = mro["mro_rate"].diff()
    ev = delta[delta.fillna(0) != 0]
    events = pd.DataFrame(index=ev.index)
    events["change"] = np.sign(ev.values).astype(int)

    print(f"[LABELS] changes: total={len(events)} | hikes={(events['change']==1).sum()} "
          f"| cuts={(events['change']==-1).sum()} | span={mro.index.min().date()}→{mro.index.max().date()}")
    return events


# === ALIGNEMENT PAR FENÊTRE (priorité futur), + MERGE-ASOF DE SECOURS ===
def align_statements_and_labels(statements: pd.DataFrame,
                                events: pd.DataFrame,
                                back_bdays=1,
                                fwd_bdays=2):
    """
    Pour chaque statement S, cherche un événement effectif E dans [S - back_bdays*BD, S + fwd_bdays*BD].
    Priorité au futur (S, S+1BD, S+2BD), sinon passé (S-1BD, S-2BD).
    Si rien trouvé, on tente un merge_asof (tolérance calendrier ±2 jours) comme secours.
    """
    from pandas.tseries.offsets import BDay

    assert isinstance(statements.index, pd.DatetimeIndex)
    assert isinstance(events.index, pd.DatetimeIndex)

    statements = statements.sort_index()
    events = events.sort_index()

    s_idx = statements.index
    ev_idx = pd.DatetimeIndex(events.index)

    forward_lags = [0] + list(range(1, fwd_bdays+1))
    backward_lags = list(range(1, back_bdays+1))

    matches_idx = []
    matches_change = []

    for s in s_idx:
        found = False
        # futur / présent
        for k in forward_lags:
            tgt = s if k == 0 else s + BDay(k)
            if tgt in ev_idx:
                matches_idx.append(s)
                matches_change.append(int(events.loc[tgt, "change"]))
                found = True
                break
        if found:
            continue
        # passé
        for k in backward_lags:
            tgt = s - BDay(k)
            if tgt in ev_idx:
                matches_idx.append(s)
                matches_change.append(int(events.loc[tgt, "change"]))
                found = True
                break

    df_lbl = statements.loc[matches_idx].copy() if matches_idx else statements.iloc[0:0].copy()
    if matches_idx:
        df_lbl["change"] = matches_change

    # Si couverture faible, tente un secours merge_asof sur calendrier civil (±2 jours)
    coverage = (len(df_lbl) / len(statements) * 100) if len(statements) else 0.0
    if coverage < 1.0 and len(events):
        # asof: on préfère l'événement le plus proche dans le futur (direction='forward') avec tolérance 2 jours
        left = statements.assign(_s=statements.index).reset_index(drop=True)
        right = events.assign(_e=events.index).reset_index(drop=True)
        merged = pd.merge_asof(
            left.sort_values("_s"), right.sort_values("_e"),
            left_on="_s", right_on="_e",
            direction="forward", tolerance=pd.Timedelta("2D")
        )
        asof_hits = merged["change"].notna()
        if asof_hits.any():
            df_asof = statements.loc[merged.loc[asof_hits, "_s"].values].copy()
            df_asof["change"] = merged.loc[asof_hits, "change"].astype(int).values
            # Combine sans dupliquer
            df_lbl = pd.concat([df_lbl, df_asof]).sort_index()
            df_lbl = df_lbl[~df_lbl.index.duplicated(keep="first")]

    coverage = (len(df_lbl) / len(statements) * 100) if len(statements) else 0.0
    vc = df_lbl["change"].value_counts().to_dict() if len(df_lbl) else {}
    print(f"[ALIGN] labeled={len(df_lbl)} / {len(statements)} ({coverage:.1f}%) | class_balance={vc} | "
          f"window=[-{back_bdays}BD, +{fwd_bdays}BD] (+ asof±2D)")

    return df_lbl


In [20]:

# -----------------------------------------------------------------------------
# Optional: supervised learning for same-day EuroStoxx returns on statement text
def supervised_returns_on_text(statements: pd.DataFrame, eurostoxx_csv=EUROSTOXX_CSV):
    try:
        ret = pd.read_csv(eurostoxx_csv)
    except FileNotFoundError:
        print("EuroStoxx daily returns CSV not found; skipping market reaction model.")
        return None

    ret["date"] = pd.to_datetime(ret["date"]).dt.tz_localize(None)
    ret = ret.set_index("date").sort_index()
    # Standardize by rolling 252-day vol to avoid heteroskedasticity
    ret["ret_std"] = ret["ret"].div(ret["ret"].ewm(252).std())
    y = ret["ret_std"].dropna()

    X = statements["text"]
    idx_ = y.index.intersection(X.index)
    if len(idx_) < 10:
        print("Not enough overlap between statements and returns; skipping.")
        return None

    X, y = X.loc[idx_], y.loc[idx_]

    est = Pipeline(
        steps=[
            ("tfidf", TfidfVectorizer(
                vocabulary=None,
                ngram_range=(1, 3),
                max_features=500,
                stop_words="english",
                token_pattern=r"\\b[a-zA-Z]{3,}\\b",
            )),
            ("reg", ElasticNet(alpha=0.0075)),
        ]
    )
    est.fit(X, y)

    vocab_ = pd.Series(est.named_steps["tfidf"].vocabulary_).sort_values().index
    interpret_coef = pd.DataFrame(np.transpose(est.named_steps["reg"].coef_), index=vocab_)
    coefs_plot(interpret_coef, title="Interpreted coefficients for market-reaction model")

    lexica = {
        "positive": interpret_coef.squeeze().nlargest(n=10),
        "negative": interpret_coef.squeeze().nsmallest(n=10),
    }
    idx_sorted = (
        pd.Series(est.predict(X), index=X.index)
        .sort_values()
        .pipe(lambda x: [x.index[0], x.index[-1]])
    )
    show_text(statements.loc[idx_sorted], lexica=lexica, n=None)
    return est, interpret_coef


In [21]:


# -----------------------------------------------------------------------------
# Run (example pipeline)
if __name__ == "__main__":
    statements = load_ecb_statements()
    mro = load_ecb_mro()
    up_dates, dw_dates, labels = compute_rate_change_labels(mro)

    # --- Classification of rate-change direction from text
    logreg_est, interpret_coef = train_ecb_tfidf_logreg(statements, up_dates, dw_dates)
    coefs_plot(interpret_coef, title="ECB: Interpreted coefficients (Logistic-ENet)")

    # Alternative: ElasticNet regression on +/-1 labels
    enet_est, interpret_coef_en = train_ecb_tfidf_elasticnet(statements, up_dates, dw_dates)
    coefs_plot(interpret_coef_en, title="ECB: Interpreted coefficients (ElasticNet)")

    # --- Implied series and markers
    pred_tfidf = implied_rate_series(enet_est, statements, up_dates, dw_dates, title="ECB implied rate (TF-IDF + ENet)")

    # --- Optional: SBERT comparison
    try:
        compare_tfidf_sbert(statements, pred_tfidf)
    except Exception as e:
        print("Skipping SBERT comparison:", e)

    # --- Optional: market reactions model
    supervised_returns_on_text(statements)



[LOAD] ECB statements from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_speeches_clean_minimal.csv
[SNIFF] encoding=utf-8 | sep=',' | quotechar='"'
[READ OK] try#1 -> shape=(2939, 5)
[OK] statements: shape=(2249, 4) | dup_dates_removed=690 | range=1997-02-07 → 2025-09-30


ValueError: empty vocabulary; perhaps the documents only contain stop words